# Theme 2: RAG Application, Starter Kit

This starter kit contains a complete, working **Retrieval-Augmented Generation (RAG)** application using **Gemma 4**, Google's open-weight model. Your team's task is to adapt and extend it for the problem statement you selected (2A or 2B).

**How RAG works:**

```
 Document ──► 1. split into chunks ──► 2. convert each chunk into an embedding (a list of numbers)
                                                           │
 Question ──► 3. retrieve the chunks whose embeddings are closest to the question's embedding
                                                           │
              4. give ONLY those chunks to Gemma ──► answer with page citations
```

**Models used (both open-weight):**
- `all-MiniLM-L6-v2` (sentence-transformers) creates the embeddings. It runs inside this notebook.
- `gemma-4-26b-a4b-it` (Google) writes the answers. It is accessed through the free Google AI Studio API.

**No GPU is required.** Use the default Colab runtime (CPU).

### Obtaining a free Google AI Studio API key (one per team)
1. Go to **https://aistudio.google.com/apikey** and sign in with a Google account.
2. Select **Create API key** and copy the key.
3. In Colab, select the **key icon** in the left sidebar (**Secrets**).
4. Select **Add new secret**. Set **Name** to `GOOGLE_API_KEY` and **Value** to your key. Enable **Notebook access**.

> **Important:** do not paste the key directly into a code cell. Anyone the notebook is shared with would be able to see it.
>
> **No key yet:** the notebook still runs. The retrieval stages work normally, and the application shows the passages it found in place of a written answer.

### How to use this notebook
- Run the cells **in order, from top to bottom**, the first time (select a cell and press **Shift + Enter**).
- Comments marked **STUDENT TODO** show the intended places to make changes.
- After changing a cell, re-run that cell and any cells below it that depend on it.
- If the notebook stops working as expected, select **Runtime → Restart session** and run all cells again from the top.

## Step 0: Installation (run once)

In [ ]:
# Installs the required libraries. This takes about one minute.
!pip install -q pypdf sentence-transformers "gradio>=6,<7" google-genai

## Step 1: Settings

These values control how documents are divided, searched and answered. They are the main parameters to experiment with.

In [ ]:
# STUDENT TODO: the Gemma 4 model that writes the answers.
#   "gemma-4-26b-a4b-it"  Mixture of Experts, 26B total / about 4B active parameters. Fast. (default)
#   "gemma-4-31b-it"      Dense, 31B parameters. More capable, slower.
GEMMA_MODEL = "gemma-4-26b-a4b-it"

# STUDENT TODO: the number of words in each chunk.
#   Small chunks (50)   more precise matches, but may lose context
#   Large chunks (300)  more context, but less precise and longer prompts
CHUNK_SIZE = 120

# STUDENT TODO: the number of words shared by neighbouring chunks, so sentences are not split.
CHUNK_OVERLAP = 30

# STUDENT TODO: the number of chunks given to Gemma for each question.
TOP_K = 3

# STUDENT TODO: the rules Gemma follows when answering. The rule to use only the context prevents invented answers.
ANSWER_RULES = """You answer questions about a document using ONLY the context passages provided.
- Each passage begins with its page number, for example [Page 2].
- After each fact you use, cite the page in brackets, for example (p. 2).
- If the answer is not in the context, reply exactly: "The answer could not be found in the document."
- Keep answers concise: 2 to 4 sentences."""

TEMPERATURE = 0.2       # a low value keeps answers consistent and close to the document

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # STUDENT TODO (optional): try "BAAI/bge-small-en-v1.5"
print("Settings configured.")

Settings configured.


## Step 2: Connect to Gemma and load the embedding model

In [28]:
import os
from google import genai
from google.genai import types, errors


def _get_secret(name):
    """Reads a secret from the Colab Secrets panel (or from an environment variable)."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


GOOGLE_API_KEY = _get_secret("GOOGLE_API_KEY")
DEMO_MODE = not GOOGLE_API_KEY
client = None if DEMO_MODE else genai.Client(api_key=GOOGLE_API_KEY)


def _friendly_error(e):
    code = getattr(e, "code", None)
    if code in (400, 401, 403) and "key" in str(e).lower():
        return "Error: the GOOGLE_API_KEY was rejected. Check that the complete key is saved in Colab Secrets, then re-run this cell."
    if code == 429:
        return "Error: the free-tier usage limit has been reached. Wait about one minute and try again."
    if code == 404:
        return f"Error: the model '{GEMMA_MODEL}' was not found. Check the GEMMA_MODEL setting."
    return f"Error: the request to Gemma failed ({e})."


def generate(user_text, system_text=None, temperature=TEMPERATURE, demo_reply="(Demonstration mode response)"):
    """Sends one message to Gemma and returns its reply as text.
    In demonstration mode, returns `demo_reply`. If the request fails, returns a message beginning with "Error:"."""
    if DEMO_MODE:
        return demo_reply
    try:
        response = client.models.generate_content(
            model = GEMMA_MODEL,
            contents = user_text,
            config = types.GenerateContentConfig(
                system_instruction = system_text,
                temperature = temperature,
                thinking_config = types.ThinkingConfig(thinking_level="minimal"),  # faster replies
            ),
        )
        return (response.text or "").strip()
    except errors.APIError as e:
        return _friendly_error(e)
    except Exception as e:
        return f"Error: {e}"


if DEMO_MODE:
    print("Demonstration mode: no GOOGLE_API_KEY was found. Retrieval will work; answers will show the passages found.")
else:
    print(f"Connected to Google AI Studio. Model: {GEMMA_MODEL}")
    print(generate("In one sentence, what is an open-weight AI model?"))

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL_NAME)   # runs on the CPU
print(f"Loaded embedding model: {EMBED_MODEL_NAME}")

Connected to Google AI Studio. Model: gemma-4-26b-a4b-it
An open-weight AI model is a machine learning model whose trained parameters are made publicly available, allowing developers to download, run, and fine-tune the model on their own hardware.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2


## Step 3: Sample document
A short, fictional course syllabus is provided so the application works immediately. Each item in the list is one page. Your own PDF can be uploaded in the application in Step 5.

In [8]:

SAMPLE_PAGES = [
    """
    Project: First Contributions
    Topic: Getting Started

    First Contributions is an open-source project that helps beginners
    learn how to make their first contribution.

    A typical contribution workflow includes:
    1. Fork the repository on GitHub.
    2. Clone your fork to your computer.
    3. Create a new branch for your changes.
    4. Make your changes.
    5. Commit your changes.
    6. Push your branch to GitHub.
    7. Open a pull request for review.

    Beginners should read the official project instructions
    before contributing.
    """,

    """
    Project: First Contributions
    Topic: Pull Requests

    A pull request proposes changes to a project repository.
    Contributors should explain what they changed and why.
    Project maintainers review the changes and may request updates
    before accepting the contribution.
    """,

    """
    Project: First Contributions
    Topic: Helpful Practices

    Read the project's README and contribution guidelines.
    Make changes on a separate branch.
    Write clear commit messages.
    Explain your changes in the pull request.
    Follow the project's Code of Conduct.
    """
]

print(f"Sample document contains {len(SAMPLE_PAGES)} pages")


Sample document contains 3 pages


## Step 4: The RAG pipeline
Each function is one stage of the pipeline. This is the core of the project and the main code to extend.

In [12]:
import numpy as np
from pypdf import PdfReader

knowledge_base = {"name": None, "chunks": [], "vectors": None}  # populated by build_knowledge_base()


def read_pdf(path):
    """Returns a list of page texts from a PDF file."""
    return [page.extract_text() or "" for page in PdfReader(path).pages]


def split_into_chunks(pages):
    """STAGE 1: split each page into overlapping chunks of about CHUNK_SIZE words."""
    chunks = []
    step = max(1, CHUNK_SIZE - CHUNK_OVERLAP)
    for page_number, text in enumerate(pages, start=1):
        words = text.split()
        for start in range(0, len(words), step):
            chunk_words = words[start:start + CHUNK_SIZE]
            if len(chunk_words) > 5:  # skip very short fragments
                chunks.append({"page": page_number, "text": " ".join(chunk_words)})
            if start + CHUNK_SIZE >= len(words):
                break
    return chunks


def build_knowledge_base(pages, name):
    """STAGE 2: convert each chunk into an embedding."""
    chunks = split_into_chunks(pages)
    if not chunks:
        raise ValueError("No text was found. The PDF may contain scanned images only.")
    vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
    knowledge_base.update(name=name, chunks=chunks, vectors=vectors)
    return f"Loaded **{name}**: {len(pages)} pages, {len(chunks)} chunks."


def retrieve(question, top_k=TOP_K):
    """STAGE 3: find the chunks whose meaning is closest to the question."""
    question_vector = embedder.encode([question], normalize_embeddings=True)[0]
    similarities = knowledge_base["vectors"] @ question_vector   # 1.0 = identical meaning
    best = np.argsort(similarities)[::-1][:top_k]
    return [{**knowledge_base["chunks"][i], "score": float(similarities[i])} for i in best]


def answer(question):
    """STAGE 4: give the retrieved chunks to Gemma and ask it to answer using only them."""
    found = retrieve(question)
    context = "\n\n".join(f"[Page {c['page']}] {c['text']}" for c in found)
    prompt = f"Context:\n{context}\n\nQuestion: {question}"
    demo = (f"**Demonstration mode:** no API key is set, so Gemma did not write an answer. Retrieval completed; "
            f"the most relevant passage is on **page {found[0]['page']}**:\n\n> {found[0]['text'][:400]}...")
    reply = generate(prompt, system_text=ANSWER_RULES, demo_reply=demo)
    return reply, found


# Quick test with the sample document.
print(build_knowledge_base(SAMPLE_PAGES, "Sample CS 101 syllabus"))
test_question = "What happens if I submit a lab late?"
reply, found = answer(test_question)
print(f"\nQuestion: {test_question}\nAnswer: {reply}\n\nRetrieved passages:")
for c in found:
    print(f"   [page {c['page']}, similarity {c['score']:.2f}] {c['text'][:90]}...")

Loaded **Sample CS 101 syllabus**: 3 pages, 3 chunks.

Question: What happens if I submit a lab late?
Answer: **Demonstration mode:** no API key is set, so Gemma did not write an answer. Retrieval completed; the most relevant passage is on **page 3**:

> Project: First Contributions Topic: Helpful Practices Read the project's README and contribution guidelines. Make changes on a separate branch. Write clear commit messages. Explain your changes in the pull request. Follow the project's Code of Conduct....

Retrieved passages:
   [page 3, similarity 0.12] Project: First Contributions Topic: Helpful Practices Read the project's README and contri...
   [page 1, similarity 0.11] Project: First Contributions Topic: Getting Started First Contributions is an open-source ...
   [page 2, similarity 0.05] Project: First Contributions Topic: Pull Requests A pull request proposes changes to a pro...


## Step 5: Launch the application
Run this cell to open the application below the cell and create a **public link** (`https://....gradio.live`). Re-run it after changing any cell above.

In [32]:
import gradio as gr

gr.close_all()  # closes any previous version of the application that is still running


APP_TITLE = "Open-Source Contributor Guide"
APP_DESCRIPTION = "Ask questions about your first open-source contribution. Get answers from the First Contributions documentation with supporting passages."

def on_upload(file):
    if file is None:
        return "No file uploaded."

    try:
        filename = os.path.basename(file)
        extension = os.path.splitext(filename)[1].lower()

        if extension == ".pdf":
            pages = read_pdf(file)

        elif extension in [".md", ".txt"]:
            with open(file, "r", encoding="utf-8") as f:
                text = f.read()

            pages = [
                section.strip()
                for section in text.split("\n## ")
                if section.strip()
            ]

        else:
            raise ValueError("Unsupported file type.")

        return build_knowledge_base(pages, filename)

    except Exception as e:
        raise gr.Error(f"The document could not be read: {e}")


def on_ask(question):
    if not question.strip():
        raise gr.Error("Enter a question first.")
    reply, found = answer(question)
    sources = "\n\n".join(f"**Page {c['page']}** (similarity {c['score']:.2f})\n> {c['text']}" for c in found)
    return reply, sources


with gr.Blocks(title=APP_TITLE) as demo:
    gr.Markdown(f"# {APP_TITLE}\n{APP_DESCRIPTION}")
    with gr.Row():
        pdf_input = gr.File(label="Upload a PDF", file_types=[".pdf"], type="filepath")
        with gr.Column():
            status_md = gr.Markdown(f"Current document: **{knowledge_base['name']}**")
            sample_btn = gr.Button("Load First Contributions Guide")
    question_box = gr.Textbox(label="Question", placeholder="For example: How is the course graded?")
    ask_btn = gr.Button("Ask", variant="primary")
    gr.Examples(
    [
        "How do I fork the First Contributions repository?",
        "How do I clone my forked repository?",
        "How do I create a pull request?",
        "What should I do if I am uncomfortable using the command line?"
    ],
    inputs=question_box
)
    answer_md = gr.Markdown()
    with gr.Accordion("Retrieved passages (the text given to Gemma)", open=False):
        sources_md = gr.Markdown()

    pdf_input = gr.File(
    label="Upload a document (PDF, Markdown, or TXT)",
    file_types=[".pdf", ".md", ".txt"],
    type="filepath"
)
    sample_btn.click(
    lambda: build_knowledge_base(README_PAGES, "First Contributions README"),
    outputs=status_md
)
    ask_btn.click(on_ask, inputs=question_box, outputs=[answer_md, sources_md])
    question_box.submit(on_ask, inputs=question_box, outputs=[answer_md, sources_md])

demo.launch(share=True)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://225c35675be232bf6a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Next steps
1. Re-read your selected problem statement (2A or 2B) in the Theme 2 document.
2. Replace the sample document and example questions with material for your problem statement.
3. Adapt `ANSWER_RULES`, the settings and the application to meet the requirements.
4. Measure the quality of the answers (see "Expected Deliverables" in the problem statement).
5. Attempt items from the **Good to have** section of the problem statement for additional credit.

**Pointers for common extensions:**
- **Markdown and text files (needed for 2B):** read them with `open(path, encoding="utf-8").read()` and treat each file (or each section under a heading) as one page, storing the file name for citations. Allow them in the application with `file_types=[".pdf", ".md", ".txt"]`.
- **Multiple documents:** use `gr.File(file_count="multiple")`, combine the pages of every file, and store the file name in each chunk for citations.
- **Answer without RAG (for comparison):** call `generate(question)` with no context and show the result next to the RAG answer.
- **Relevance threshold:** in `answer()`, if `found[0]["score"]` is below a chosen value, return "The answer could not be found in the document." without calling Gemma.
- **Conversation memory:** replace the question box with `gr.ChatInterface` and include recent messages in the prompt.

## Troubleshooting

| Issue | Resolution |
|---|---|
| The notebook remains in demonstration mode | Add `GOOGLE_API_KEY` in Secrets, enable **Notebook access**, and re-run the Step 2 cell. |
| "API key was rejected" | Create a new key at aistudio.google.com/apikey and update the secret. |
| "Usage limit reached" | Wait about one minute. Free keys allow a limited number of requests per minute. Team members may each create a key. |
| `NameError: name '...' is not defined` | A cell was skipped. Run all cells above in order. |
| "No text was found" | The PDF contains scanned images only. Use a text-based PDF. |
| The public link no longer works | Links expire when the Colab session disconnects. Re-run the launch cell to create a new link. |

# ## Download First Contributions README

In [16]:
import requests

url = "https://raw.githubusercontent.com/firstcontributions/first-contributions/main/README.md"

response = requests.get(url)
response.raise_for_status()

readme_text = response.text

print(readme_text[:1000])

[![Open Source Love](https://firstcontributions.github.io/open-source-badges/badges/open-source-v1/open-source.svg)](https://github.com/firstcontributions/open-source-badges)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)
[![Open Source Helpers](https://www.codetriage.com/roshanjossey/first-contributions/badges/users.svg)](https://www.codetriage.com/roshanjossey/first-contributions)

#### _Read this in [other languages](docs/translations/Translations.md)._
<kbd>[<img title="Shqip" alt="Shqip" src="https://cdn.jsdelivr.net/gh/hampusborgos/country-flags@main/svg/al.svg" width="22">](docs/translations/README.sq.md)</kbd>
<kbd>[<img title="Armenian" alt="Armenian" src="https://cdn.jsdelivr.net/gh/hampusborgos/country-flags@main/svg/am.svg" width="22">](docs/translations/README.arm.md)</kbd>
<kbd>[<img title="Uzbek" alt="Uzbek language" src="https://cdn.jsdelivr.net/gh/hampusborgos/country-flags@main/svg/uz.svg" width="22">](docs/t

## Display First Contributions README Content

In [ ]:
print("README downloaded:", len(readme_text), "characters")
print("\nFirst Contributions guide:\n")
print(readme_text[:3000])

## Prepare First Contributions Documentation for RAG

In [17]:
# Check that the README was downloaded correctly
print("README length:", len(readme_text), "characters")
print("\nREADME preview:\n")
print(readme_text[:1500])

README length: 25012 characters

README preview:

[![Open Source Love](https://firstcontributions.github.io/open-source-badges/badges/open-source-v1/open-source.svg)](https://github.com/firstcontributions/open-source-badges)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)
[![Open Source Helpers](https://www.codetriage.com/roshanjossey/first-contributions/badges/users.svg)](https://www.codetriage.com/roshanjossey/first-contributions)

#### _Read this in [other languages](docs/translations/Translations.md)._
<kbd>[<img title="Shqip" alt="Shqip" src="https://cdn.jsdelivr.net/gh/hampusborgos/country-flags@main/svg/al.svg" width="22">](docs/translations/README.sq.md)</kbd>
<kbd>[<img title="Armenian" alt="Armenian" src="https://cdn.jsdelivr.net/gh/hampusborgos/country-flags@main/svg/am.svg" width="22">](docs/translations/README.arm.md)</kbd>
<kbd>[<img title="Uzbek" alt="Uzbek language" src="https://cdn.jsdelivr.net/gh/hampusborgos/

## Load README into the RAG Knowledge Base

In [18]:
# Split the README into manageable sections
README_PAGES = [
    section.strip()
    for section in readme_text.split("\n## ")
    if section.strip()
]

print("README sections loaded:", len(README_PAGES))
print("\nFirst section preview:\n")
print(README_PAGES[0][:1000])

README sections loaded: 9

First section preview:

[![Open Source Love](https://firstcontributions.github.io/open-source-badges/badges/open-source-v1/open-source.svg)](https://github.com/firstcontributions/open-source-badges)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)
[![Open Source Helpers](https://www.codetriage.com/roshanjossey/first-contributions/badges/users.svg)](https://www.codetriage.com/roshanjossey/first-contributions)

#### _Read this in [other languages](docs/translations/Translations.md)._
<kbd>[<img title="Shqip" alt="Shqip" src="https://cdn.jsdelivr.net/gh/hampusborgos/country-flags@main/svg/al.svg" width="22">](docs/translations/README.sq.md)</kbd>
<kbd>[<img title="Armenian" alt="Armenian" src="https://cdn.jsdelivr.net/gh/hampusborgos/country-flags@main/svg/am.svg" width="22">](docs/translations/README.arm.md)</kbd>
<kbd>[<img title="Uzbek" alt="Uzbek language" src="https://cdn.jsdelivr.net/gh/hampusborgos

## Prepare README Data for RAG Retrieval

In [ ]:
README_PAGES = [
    section.strip()
    for section in readme_text.split("\n## ")
    if section.strip()
]

print("Number of sections:", len(README_PAGES))
print("\nFirst section:\n")
print(README_PAGES[0][:500])

## Load First Contributions README into RAG

In [20]:
build_knowledge_base(README_PAGES, "First Contributions README")

'Loaded **First Contributions README**: 9 pages, 17 chunks.'

## Test Retrieval with Real Documentation

In [21]:
question = "How do I contribute to an open-source project?"

reply, found = answer(question)

print("Answer:")
print(reply)

print("\nRetrieved sources:")
for c in found:
    print(c["text"][:300])
    print("-" * 40)

Answer:
**Demonstration mode:** no API key is set, so Gemma did not write an answer. Retrieval completed; the most relevant passage is on **page 8**:

> Where to go from here? Congrats! You just completed the standard _fork -> clone -> edit -> pull request_ workflow that you'll often encounter as a contributor! Celebrate your contribution and share it with your friends and followers by going to [web app](https://firstcontributions.github.io/#social-share). If you'd like more practice, checkout [code contributions](https://github.com/roshanjossey/c...

Retrieved sources:
Where to go from here? Congrats! You just completed the standard _fork -> clone -> edit -> pull request_ workflow that you'll often encounter as a contributor! Celebrate your contribution and share it with your friends and followers by going to [web app](https://firstcontributions.github.io/#social-
----------------------------------------
<kbd>[<img title="Lingala" alt="Lingala" src="https://cdn.jsdelivr.net/gh/hampusb

## Clean README Content for Better Retrieval

In [22]:
import re

clean_readme = re.sub(r'<[^>]+>', ' ', readme_text)
clean_readme = re.sub(r'!\[[^\]]*\]\([^)]*\)', ' ', clean_readme)
clean_readme = re.sub(r'\[([^\]]+)\]\([^)]*\)', r'\1', clean_readme)
clean_readme = re.sub(r'\s+', ' ', clean_readme).strip()

README_PAGES = [
    section.strip()
    for section in clean_readme.split("## ")
    if section.strip()
]

print("Clean sections:", len(README_PAGES))
print(README_PAGES[0][:500])

Clean sections: 13
##


## Check the Cleaned README Content

In [23]:
print(repr(README_PAGES[0][:500]))

'##'


## Split README into Clean Sections

In [24]:
README_PAGES = [
    section.strip()
    for section in clean_readme.split("## ")
    if section.strip() and section.strip() != "##"
]

print("Number of sections:", len(README_PAGES))
print("First section:")
print(README_PAGES[0][:500])

Number of sections: 12
First section:
_Read this in other languages._ # First Contributions This project aims to simplify and guide the way beginners make their first contribution. If you are looking to make your first contribution, follow the steps below. _If you're not comfortable with command line, here are tutorials using GUI tools._ ##


## Final Cleanup of README Sections

In [25]:
README_PAGES = [
    section.replace("##", "").strip()
    for section in README_PAGES
    if section.replace("##", "").strip()
]

print("Clean sections:", len(README_PAGES))
print("\nFirst section:")
print(README_PAGES[0][:500])

Clean sections: 12

First section:
_Read this in other languages._ # First Contributions This project aims to simplify and guide the way beginners make their first contribution. If you are looking to make your first contribution, follow the steps below. _If you're not comfortable with command line, here are tutorials using GUI tools._


## Reload RAG with Clean README Data

In [26]:
build_knowledge_base(README_PAGES, "First Contributions README")

'Loaded **First Contributions README**: 12 pages, 11 chunks.'

## Test RAG with Clean Documentation

In [27]:
question = "What are the steps to make my first open-source contribution?"

reply, found = answer(question)

print("Answer:")
print(reply)

print("\nRetrieved sources:")
for c in found:
    print(c["text"][:400])
    print("-" * 40)

Answer:
**Demonstration mode:** no API key is set, so Gemma did not write an answer. Retrieval completed; the most relevant passage is on **page 1**:

> _Read this in other languages._ # First Contributions This project aims to simplify and guide the way beginners make their first contribution. If you are looking to make your first contribution, follow the steps below. _If you're not comfortable with command line, here are tutorials using GUI tools._...

Retrieved sources:
_Read this in other languages._ # First Contributions This project aims to simplify and guide the way beginners make their first contribution. If you are looking to make your first contribution, follow the steps below. _If you're not comfortable with command line, here are tutorials using GUI tools._
----------------------------------------
Where to go from here? Congrats! You just completed the standard _fork -> clone -> edit -> pull request_ workflow that you'll often encounter as a contributor! Celebrate your cont

## Configure Gemma API Key

## Test Gemma with First Contributions Documentation

In [29]:
question = "What are the steps to make my first open-source contribution?"

reply, found = answer(question)

print("Answer:")
print(reply)

print("\nRetrieved sources:")
for c in found:
    print(c["text"][:400])
    print("-" * 40)

Answer:
To make your first contribution, you should follow the standard fork -> clone -> edit -> pull request workflow (p. 10). You can begin by clicking the fork button at the top of the page to create a copy of the repository in your account (p. 3). If you are uncomfortable with the command line, tutorials using GUI tools are available (p. 1).

Retrieved sources:
_Read this in other languages._ # First Contributions This project aims to simplify and guide the way beginners make their first contribution. If you are looking to make your first contribution, follow the steps below. _If you're not comfortable with command line, here are tutorials using GUI tools._
----------------------------------------
Where to go from here? Congrats! You just completed the standard _fork -> clone -> edit -> pull request_ workflow that you'll often encounter as a contributor! Celebrate your contribution and share it with your friends and followers by going to web app. If you'd like more practice, checko

## Test Questions Outside the Documentation

In [30]:
question = "Who won the 2022 FIFA World Cup?"

reply, found = answer(question)

print("Answer:")
print(reply)

print("\nRetrieved passages found:", len(found))

Answer:
The answer could not be found in the document.

Retrieved passages found: 3


## Verify Answers Against the Source Document

In [31]:
question = "How do I fork the First Contributions repository?"

reply, found = answer(question)

print("Answer:")
print(reply)

print("\nRetrieved passages:")
for c in found:
    print(c["text"][:300])
    print("-" * 40)

Answer:
To fork the repository, click on the fork button located at the top of the page (p. 3). This action will create a copy of the repository in your account (p. 3).

Retrieved passages:
Fork this repository Fork this repository by clicking on the fork button on the top of this page. This will create a copy of this repository in your account.
----------------------------------------
Clone the repository Now clone the forked repository to your machine. Go to your GitHub account, open the forked repository, click on the code button, then on SSH tab and then click the _copy url to clipboard_ icon. Open a terminal and run the following git command: ```bash git clone "url you just c
----------------------------------------
Where to go from here? Congrats! You just completed the standard _fork -> clone -> edit -> pull request_ workflow that you'll often encounter as a contributor! Celebrate your contribution and share it with your friends and followers by going to web app. If you'd like m

In [33]:
demo.launch(share=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://225c35675be232bf6a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# New section